In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# Load data
df = pd.read_csv("zameen-updated.csv")

# Clean data
df_sale = df[df['purpose'] == 'For Sale'].copy()
df_sale = df_sale[df_sale['price'] > 0].copy()
df_sale = df_sale[df_sale['property_type'].isin(['House', 'Flat'])].copy()

# Convert Kanal to Marla
def convert_to_marla(row):
    if row['Area Type'] == 'Kanal':
        return row['Area Size'] * 20
    else:
        return row['Area Size']

df_sale['area_marla'] = df_sale.apply(convert_to_marla, axis=1)

# Further cleaning
df_sale = df_sale[df_sale['price'] > 100000].copy()
df_sale = df_sale[(df_sale['bedrooms'] > 0) & (df_sale['baths'] > 0)].copy()
df_sale = df_sale[df_sale['area_marla'] < 500].copy()

# Log transforms
df_sale['log_price'] = np.log1p(df_sale['price'])
df_sale['log_area'] = np.log1p(df_sale['area_marla'])

# Encode categories
le_city = LabelEncoder()
le_type = LabelEncoder()
df_sale['city_encoded'] = le_city.fit_transform(df_sale['city'])
df_sale['property_type_encoded'] = le_type.fit_transform(df_sale['property_type'])

# Prepare features
features = ['log_area', 'bedrooms', 'baths', 'city_encoded', 'property_type_encoded', 'latitude', 'longitude']
X = df_sale[features]
y = df_sale['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Train Random Forest (compact version for deployment)
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
print("Random Forest R² Score:", r2_score(y_test, rf_predictions))
print("Random Forest MAE (log scale):", mean_absolute_error(y_test, rf_predictions))

# Save model and encoders
joblib.dump(rf_model, 'house_price_model.pkl', compress=3)
joblib.dump(le_city, 'city_encoder.pkl')
joblib.dump(le_type, 'property_type_encoder.pkl')

print("Model and encoders saved successfully!")

Training set: (71713, 7)
Test set: (17929, 7)
Random Forest R² Score: 0.9163925864806122
Random Forest MAE (log scale): 0.17129173805668235
Model and encoders saved successfully!
